# AKD-EXT Closed Loop FM_Prithvi Playground

## Closed loop workflow - FM_Prithvi

A pipeline of FM_Prithvi-specialized closed-loop agents. These agents are designed to work sequentially, where each stage's output feeds into the next.

**Required env vars:** `OPENAI_API_KEY`

| Stage | Agent | Purpose |
|-------|-------|---------|
| 1 | FMPrithviGapAgent | Detects research gaps + candidate RQs from a paper corpus |
| 2 | FMPrithviCapabilityFeasibilityMapperAgent | Maps RQ requirements to FM_Prithvi pipeline capabilities; Go / Conditional-Go / No-Go + Handoff Package |
| 3 | FMPrithviWorkflowSpecBuilderAgent | Builds workflow spec + executor pipeline config YAML |

> The Prithvi system prompts include a "single-shot execution mode" override at the bottom that auto-confirms every PAUSE checkpoint when the agent is invoked via `arun()`. So each stage produces its complete structured artifact in one call. If you want to drive a stage interactively instead, override `system_prompt` on its config to remove the override block.

### Stage 1: Gap Agent

Provide a corpus of academic papers (extracted text or summaries) and the agent will produce a markdown report with ranked gaps, contradictions, and candidate research questions. The report flows directly into Stage 2.

In [1]:
from akd_ext.agents.closed_loop.prithvi import (
    FMPrithviGapAgent,
    FMPrithviGapAgentConfig,
)
from akd_ext.agents.gap import GapAgentInputSchema

/Users/rsahoo/Desktop/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# Stage 1 input: paper corpus
# TODO: replace with actual extracted paper text or summaries.
# Use stable PaperIDs (e.g. P01, P02) so the agent's traceability fields can reference them.

corpus = """# Corpus

## P01 — <Paper Title>
Authors: ...
Year: ...

<Full text or detailed summary>

## P02 — <Paper Title>
Authors: ...
Year: ...

<Full text or detailed summary>
"""

In [ ]:
# Stage 1: Gap Agent
gap_agent = FMPrithviGapAgent(FMPrithviGapAgentConfig())

gap_result = await gap_agent.arun(
    GapAgentInputSchema(query=corpus)
)

print(gap_result.report)

In [ ]:
# Save the gap report; downstream Stage 2 reads this as research_questions_md
from pathlib import Path

gap_md_path = Path("prithvi_gap_report.md")
gap_md_path.write_text(gap_result.report, encoding="utf-8")
print(f"Saved gap report to: {gap_md_path}")

research_question = gap_result.report

In [3]:
research_question = """
I prioritized **all 8** into qualitative tiers using the allowed criteria: **conceptual value, intra-corpus novelty, and likely impact within this corpus**, while **excluding feasibility**.

No numeric scoring. No forced ordering within tiers.

## Tier 1 — High
These are the strongest problem-driven candidates in this corpus because they sit on repeated cross-paper tensions, not just single-study limitations.

### RQ2
**Research question**  
Under what conditions does combining flood extent with crop-condition time series improve crop-specific flood-damage assessment relative to inundation-only approaches?

**Candidate H₀ / H₁ or neutral framing**  
- **H₀:** Adding crop-condition time series does not materially improve crop-specific flood-damage assessment over inundation-only approaches.  
- **H₁:** Combining flood extent with crop-condition time series improves crop-specific flood-damage assessment, but the magnitude of improvement varies by crop, timing, and flood context.

**Variables / proxies**  
- Flood extent / inundation status  
- Vegetation-condition trajectory or index change  
- Crop type  
- Damage class or relative loss class

**Context constraints**  
- Spatial scope: flood-affected croplands in event-based case studies  
- Temporal scope: immediate post-event window plus short recovery window  
- Conditions: cloud-contaminated flood periods where optical-only assessment may be unstable

**Linked gap(s)**  
- Gap 2  
- Gap 4  
- Gap 5

**Causality guardrails**  
Do not treat VI decline as direct yield loss without independent validation. Prioritize comparative performance framing.

**Confidence**  
**High** — Strongly grounded in the review and flood case studies that repeatedly argue flood extent alone is insufficient.【18:0†2020_A Systematic Review on Case Studies of Remote-Sensing-Based Flood Crop Loss Assessment.pdf†L1-L18】【19:0†2021_Remote Sensing Based Rapid Assessment of Flood Crop Damage Using Novel Disaster Vegetation Damage Index.pdf†L1-L18】【20:0†2023_A Comprehensive Evaluation of Flooding's Effect on Crops Using Satellite Time Series Data.pdf†L1-L18】

**Why this tier**  
This is the clearest cross-paper scientific problem in the flood subset and directly addresses the strongest explicit gap signal in the corpus.

---

### RQ5
**Research question**  
What minimum validation design is needed to make crop-specific disturbance assessments transferable across regions or events?

**Candidate H₀ / H₁ or neutral framing**  
- **Neutral hypothesis framing:** Validation requirements differ by disturbance type and target variable, but some minimum combination of independent reference data, scale-matched labels, and multi-context testing is needed for transferability claims.

**Variables / proxies**  
- Validation sample size  
- Validation source type: field survey, high-resolution image reference, yield data, administrative records  
- Spatial and temporal coverage of validation  
- Cross-region or cross-event performance stability

**Context constraints**  
- Spatial scope: multiple regions or events within either burning or flooding, ideally both  
- Temporal scope: more than one season or event year  
- Conditions: heterogeneous crop systems and sensor availability

**Linked gap(s)**  
- Gap 3  
- Gap 1  
- Gap 2

**Causality guardrails**  
This is a validation-design question, not a causal impact question.

**Confidence**  
**High** — Validation scarcity is one of the most repeated limitations across both burn and flood papers.【17:0†2009_The spatial and temporal distribution of crop residue burning in the contiguous United States.pdf†L1-L21】【18:0†2020_A Systematic Review on Case Studies of Remote-Sensing-Based Flood Crop Loss Assessment.pdf†L1-L18】【23:0†2024_glocab_global cropland burned area from mid-2002 to 2020.pdf†L1-L18】

**Why this tier**  
It has strong cross-hazard relevance and directly targets a corpus-wide bottleneck that currently prevents stronger comparison or generalization.

---

### RQ6
**Research question**  
Do crop-specific qualitative flood-damage classes align consistently with independent yield-loss information across crop types and flood events?

**Candidate H₀ / H₁ or neutral framing**  
- **H₀:** Qualitative flood-damage classes are not consistently associated with independent yield-loss information across crop types and events.  
- **H₁:** Qualitative flood-damage classes are associated with yield loss, but the strength and consistency of the association vary by crop and event.

**Variables / proxies**  
- Qualitative damage class  
- Yield-loss percentage or other production-loss proxy  
- Crop type  
- Event identity

**Context constraints**  
- Spatial scope: multiple flood case studies with some form of independent production or yield data  
- Temporal scope: event year plus corresponding harvest outcome period  
- Conditions: crop-specific analysis rather than generic cropland

**Linked gap(s)**  
- Gap 2  
- Gap 3  
- Contradiction 1

**Causality guardrails**  
Association-first. A useful class system need not be a precise yield model.

**Confidence**  
**High** — This directly tests one of the sharpest internal disagreements in the corpus: rapid qualitative usefulness versus insufficient quantitative loss inference.【18:0†2020_A Systematic Review on Case Studies of Remote-Sensing-Based Flood Crop Loss Assessment.pdf†L1-L18】【19:0†2021_Remote Sensing Based Rapid Assessment of Flood Crop Damage Using Novel Disaster Vegetation Damage Index.pdf†L1-L18】

**Why this tier**  
It converts a live contradiction into a concrete comparative question with strong interpretive payoff inside the corpus.

## Tier 2 — Medium
These are strong and defensible, but either somewhat broader, more method-comparative, or less sharply constrained by corpus disagreement.

### RQ1
**Research question**  
How does cropland-burned-area detection change when disturbance is estimated from coarse active-fire products versus higher-resolution optical reference imagery across small-field agricultural landscapes?

**Candidate H₀ / H₁ or neutral framing**  
- **H₀:** Coarse active-fire-based cropland burn estimates do not differ systematically from higher-resolution reference estimates in small-field agricultural regions.  
- **H₁:** Coarse active-fire-based estimates systematically underdetect cropland burning relative to higher-resolution reference estimates in small-field agricultural regions.

**Variables / proxies**  
- Burned / not-burned cropland area  
- Field size or small-field landscape context  
- Sensor/product source  
- Detection disagreement / omission rate

**Context constraints**  
- Spatial scope: small-field cropland regions, especially settings like Arkansas or other intensively managed agricultural areas  
- Temporal scope: post-harvest burning windows within one or more burn seasons  
- Conditions: short-duration agricultural burns, rapid tillage or residue removal

**Linked gap(s)**  
- Gap 1  
- Gap 3

**Causality guardrails**  
Association and measurement-comparison first. Do not assume that product disagreement alone proves one sensor is universally correct.

**Confidence**  
**High** — Repeatedly grounded in the burn subset from CONUS to county to global product comparisons.【17:0†2009_The spatial and temporal distribution of crop residue burning in the contiguous United States.pdf†L1-L21】【22:0†2023_Crop Residue burning from high-resolution satellite imagery and PM2.5 dispersion  A case study of Mississippi County  Arkansas  USA.pdf†L1-L19】【23:0†2024_glocab_global cropland burned area from mid-2002 to 2020.pdf†L1-L18】

**Why this tier**  
Very solid, but somewhat narrower than Tier 1 because it is concentrated in the burning subset and the general underdetection premise is already well signaled in the corpus.

---

### RQ4
**Research question**  
How much crop-type detail is necessary for reliable disturbance assessment at different spatial scales?

**Candidate H₀ / H₁ or neutral framing**  
- **H₀:** Broad cropland masks perform comparably to crop-type-specific attribution for disturbance assessment across scales.  
- **H₁:** The required level of crop-type detail increases as the assessment target moves from generic extent mapping toward damage, recovery, production loss, or emissions estimation.

**Variables / proxies**  
- Crop attribution level: generic cropland, broad crop groups, specific crop classes  
- Assessment target: extent, severity, recovery, production loss, emissions  
- Spatial scale: local, regional, national, global  
- Performance / stability measure

**Context constraints**  
- Spatial scope: multi-scale comparison across local case studies and broader products  
- Temporal scope: event windows or seasonal burning periods  
- Conditions: systems with crop rotation, double-cropping, or heterogeneous planting calendars

**Linked gap(s)**  
- Gap 4  
- Gap 1  
- Gap 2

**Causality guardrails**  
Frame as methodological sufficiency, not causal crop vulnerability.

**Confidence**  
**Medium-High** — Present across both hazards, especially where static crop maps or simplified crop groupings are flagged as limits.【21:0†2023_A framework for multi-sensor satellite data to evaluate crop production losses_the case study of 2022 Pakistan floods.pdf†L1-L18】【23:0†2024_glocab_global cropland burned area from mid-2002 to 2020.pdf†L1-L18】

**Why this tier**  
Useful as a cross-scale synthesis question, but broader and slightly less sharply anchored than the top Tier 1 questions.

---

### RQ7
**Research question**  
How should crop-production-loss frameworks distinguish direct inundation impacts from concurrent rainfall, drainage, or management-related crop damage pathways during major flood disasters?

**Candidate H₀ / H₁ or neutral framing**  
- **Neutral hypothesis framing:** Major-flood crop loss reflects multiple concurrent pathways, and treating inundation as the sole driver will misattribute some losses.  
- **Alternative directional framing:** Explicit separation of inundation and non-inundation pathways improves interpretation of crop-production-loss estimates.

**Variables / proxies**  
- Inundation extent  
- Rainfall intensity or anomaly  
- Crop condition change  
- Management or drainage-context proxy  
- Production-loss estimate

**Context constraints**  
- Spatial scope: large flood disasters affecting broad agricultural regions  
- Temporal scope: event period plus early post-event damage window  
- Conditions: complex flood settings with flash flooding, standing water, or infrastructure/management deficiencies

**Linked gap(s)**  
- Gap 2  
- Contradiction 2

**Causality guardrails**  
Avoid attributing all observed crop loss to floodwater extent without pathway separation.

**Confidence**  
**Medium-High** — Strongly motivated by the Pakistan study and consistent with the broader critique of inundation-only reasoning.【21:0†2023_A framework for multi-sensor satellite data to evaluate crop production losses_the case study of 2022 Pakistan floods.pdf†L1-L18】【18:0†2020_A Systematic Review on Case Studies of Remote-Sensing-Based Flood Crop Loss Assessment.pdf†L19-L34】

**Why this tier**  
Important and specific, but more dependent on one contradiction-centered reading of the flood-production-loss subset.

---

### RQ8
**Research question**  
Are cross-scale cropland-burning estimates more stable when burned area is inferred through conversion factors linked to active fire than when it is directly mapped from coarse burned-area products alone?

**Candidate H₀ / H₁ or neutral framing**  
- **H₀:** Conversion-factor approaches linked to active fire do not provide more stable cross-scale cropland-burn estimates than coarse burned-area products alone.  
- **H₁:** Conversion-factor approaches linked to active fire provide more stable cropland-burn estimates across scales, especially where direct coarse burned-area mapping omits small fires.

**Variables / proxies**  
- Burned-area estimate  
- Method family: coarse burned-area product alone vs active-fire plus conversion-factor approach  
- Cross-scale agreement  
- Crop-region type

**Context constraints**  
- Spatial scope: local reference areas plus regional or broader products  
- Temporal scope: repeated burn seasons or multi-year windows  
- Conditions: frequent small agricultural fires and heterogeneous field patterns

**Linked gap(s)**  
- Gap 1  
- Gap 3

**Causality guardrails**  
Method-comparison framing only.

**Confidence**  
**Medium-High** — Strongly supported inside the burn subset, especially by the GloCAB framing, but narrower and closer to an existing method family already represented in the corpus.【23:0†2024_glocab_global cropland burned area from mid-2002 to 2020.pdf†L1-L18】【17:0†2009_The spatial and temporal distribution of crop residue burning in the contiguous United States.pdf†L19-L34】

**Why this tier**  
Defensible and well tied to the corpus, though less conceptually broad than Tier 1.

## Tier 3 — Exploratory

### Correction to tiering
After checking for clearer separation, I move **RQ3** from **Tier 2** to **Tier 3** because its core resilience framing is promising but supported by fewer papers in this set than the rest of Tier 2.

### RQ3
**Research question**  
Can crop-specific flood damage be estimated more reliably from event-time damage indicators or from post-event recovery trajectories?

**Candidate H₀ / H₁ or neutral framing**  
- **Neutral hypothesis framing:** Event-time damage indicators and recovery trajectories capture different aspects of flood impact and may not rank crop damage identically.  
- **Alternative directional framing:** Recovery trajectories may better explain eventual crop outcome than event-time damage indicators alone in some flood contexts.

**Variables / proxies**  
- Event-time damage class  
- Recovery / resilience trajectory from satellite time series  
- Crop type  
- Final damage or loss proxy

**Context constraints**  
- Spatial scope: flood-affected cropland regions with repeat observations after the event  
- Temporal scope: event window through later-season recovery period  
- Conditions: crops with heterogeneous resilience and possible farmer replanting or management response

**Linked gap(s)**  
- Gap 2  
- Gap 5  
- Contradiction 1

**Causality guardrails**  
Association-first. Recovery differences may reflect management intervention, replanting, or seasonal timing rather than flood effect alone.

**Confidence**  
**Medium-High** — Well motivated by the resilience-oriented flood papers, but less repeatedly tested across the corpus.【19:0†2021_Remote Sensing Based Rapid Assessment of Flood Crop Damage Using Novel Disaster Vegetation Damage Index.pdf†L19-L34】【20:0†2023_A Comprehensive Evaluation of Flooding's Effect on Crops Using Satellite Time Series Data.pdf†L19-L34】

**Why this tier**  
Promising and potentially distinctive, but more weakly anchored across the full corpus than the higher tiers.

## Final tier structure

### High
- RQ2
- RQ5
- RQ6

### Medium
- RQ1
- RQ4
- RQ7
- RQ8

### Exploratory
- RQ3

## Reviewer-risk style notes from this corpus
- Flood case-study questions should avoid becoming documentation-only event analyses; they will need explicit generalizability framing beyond one event or one region.
- Burn-detection questions should avoid simple “method A beats method B” framing without explaining where and why disagreement arises.
- Crop-specific questions will likely attract scrutiny around validation quality, crop-map quality, and scale transfer.
"""

### Stage 2: Capability/Feasibility Mapper Agent

In [4]:
from akd_ext.agents.closed_loop.prithvi import (
    FMPrithviCapabilityFeasibilityMapperAgent,
    FMPrithviCapabilityFeasibilityMapperConfig,
)
from akd_ext.agents.closed_loop.stages.capability_feasibility_mapper import (
    CapabilityFeasibilityMapperInputSchema,
)

In [5]:
# Stage 2: Capability Feasibility Mapper
mapper_agent = FMPrithviCapabilityFeasibilityMapperAgent(FMPrithviCapabilityFeasibilityMapperConfig())

result = await mapper_agent.arun(
    CapabilityFeasibilityMapperInputSchema(
        research_questions_md=research_question,
    )
)

print(result.report)

# Capability & Feasibility Assessment (Prithvi Pipeline) — 5 Stages

**Input:** `research_questions_md` (8 RQs: RQ1–RQ8)  
**Capability inventory references:** Pipeline Capability Envelope (v2), Feasibility Mapper Full Process (v2), Ancillary Dataset Inventory (Combined), Stage 2.2 context doc (sections 4–7, 12)  
**Timestamp:** 2026-04-28  

---

## Stage 1 — Requirement Decomposition (atomic requirements per RQ)

### RQ2 (Flood extent + crop-condition time series for crop-specific damage)
| # | Dimension | Requirement | Derived From |
|---:|---|---|---|
| RQ2-1 | Model/Tool | Generate **flood extent mask** at ~30m for each event date | Variable: Flood extent / inundation status |
| RQ2-2 | Model/Tool | Generate **crop type map** (or ingest crop baseline) for crop-specific attribution | Variable: Crop type |
| RQ2-3 | Input Data | Acquire **HLS** imagery on **exact flood date** for flood mapping (S30/L30) | Context: event-based case studies |
| RQ2-4 | Input Data | Acquire **multi-tem

In [8]:
from pathlib import Path
# Saving the md file for testing purpose
report_md_path = Path("prithvi_feasibility_report.md")
report_md_path.write_text(result.report, encoding="utf-8")
print(f"Saved feasibility report to: {report_md_path}")
report_md_content = result.report

Saved feasibility report to: prithvi_feasibility_report.md


### Stage 3: Workflow Spec Builder

In [9]:
from akd_ext.agents.closed_loop.prithvi import (
    FMPrithviWorkflowSpecBuilderAgent,
    FMPrithviWorkflowSpecBuilderConfig,
)
from akd_ext.agents.closed_loop.stages.workflow_spec_builder import (
    WorkflowSpecBuilderInputSchema,
)

In [10]:
# Stage 3: Workflow Spec Builder
# Per the Stage-3 system prompt, the Stage-2 feasibility report IS the
# authoritative handoff (it embeds the locked Stage-1 RQs in Stage 5 of the
# Feasibility Mapper output).
spec_agent = FMPrithviWorkflowSpecBuilderAgent(FMPrithviWorkflowSpecBuilderConfig())

spec_result = await spec_agent.arun(
    WorkflowSpecBuilderInputSchema(
        stage_1_hypotheses=research_question,
        stage_2_feasibility=report_md_content,
    )
)

print("SPEC:")
print("=" * 80)
print(spec_result.spec)
print("\nREASONING:")
print("=" * 80)
print(spec_result.reasoning)

SPEC:
# Experiment Workflow Specification (v2) — Prithvi Closed-Loop Pipeline

**Input handoff:** Stage-2 Capability & Feasibility Assessment (timestamp **2026-04-28**)  
**This document designs workflows; it does not execute or interpret results.**  

---

## 1) Experiment Design Narrative (Stages 1–2)

### 1.1 Handoff review (Stage 1 summary; authoritative)
The feasibility handoff provides **8 locked research questions (RQ1–RQ8)** with requirement decompositions and capability mappings. Pipeline alignment is **High** for:
- **Flood**: Prithvi Flood (Tier 1), OPERA DSWx-HLS (2023+), GFM (2017+, credentials), NDVI severity (MOD13A1 + VNP13A1).
- **Burn**: Prithvi Burn (Tier 1), FIRMS (active fires), MTBS (CONUS reference), MCD64A1 (global coarse burned area).
- **Crop**: Prithvi Crop (Tier 1), regional crop baselines (CDL CONUS; EUCROPMAP+CLMS Europe; AAFC Canada).

Key feasibility-critical constraints carried forward:
1. **Flood module requires HLS on the exact flood date** (no buffer

In [12]:
# Save the workflow spec as a markdown file
spec_md_path = Path("prithvi_workflow_spec.md")
spec_md_path.write_text(
    spec_result.spec + "\n\n---\n\n# Reasoning\n\n" + spec_result.reasoning,
    encoding="utf-8",
)
print(f"Saved workflow spec to: {spec_md_path}")

Saved workflow spec to: prithvi_workflow_spec.md
